# The Goertzel Algorithm

The Goertzel Algorithm is an efficient method for computing a **single frequency bin** of the DFT. Where a full FFT costs O(N log N) to evaluate all N bins, Goertzel costs O(N) for one target frequency. It's ideal when you only need to detect or measure a known tone — DTMF detection is the classic example.

This notebook walks through the algorithm from first principles, then visualises what each stage is doing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. What is a DFT bin?

The Discrete Fourier Transform at bin $k$ is:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j 2\pi k n / N}$$

This is a **correlation** of the signal $x[n]$ with a complex exponential (a spinning phasor) at normalised frequency $\omega_k = 2\pi k / N$.

Computing this directly requires $N$ complex multiplications per bin. Goertzel recasts it as a **recursive filter** so we avoid the multiplications entirely during the accumulation loop.

## 2. Derivation — from DFT to a difference equation

Multiply $X[k]$ by $1 = e^{j 2\pi k N / N} = W_N^{-kN}$ (this equals 1 because $e^{j2\pi k} = 1$ for integer $k$):

$$X[k] = W_N^{-kN} \sum_{n=0}^{N-1} x[n] W_N^{kn} = \sum_{n=0}^{N-1} x[n] W_N^{-k(N-n)}$$

where $W_N = e^{j2\pi/N}$ and $W_N^{-k} = e^{-j2\pi k/N}$.

Define an intermediate sequence:

$$y_k[n] = \sum_{m=0}^{n} x[m] \cdot W_N^{-k(n-m)}$$

This satisfies the **recurrence**:

$$\boxed{y_k[n] = x[n] + W_N^{-k} \cdot y_k[n-1]}$$

with $y_k[-1] = 0$, and the final answer is $X[k] = y_k[N-1]$.

That recurrence *is* Goertzel's algorithm. But we can make it cheaper.

## 3. Eliminating the complex multiply

The multiplier $W_N^{-k} = e^{-j2\pi k/N}$ is complex. We can split $y_k[n]$ into a two-stage filter:

**Stage 1 — real-coefficient IIR (run for all N samples):**

$$s[n] = x[n] + 2\cos(\omega_k)\, s[n-1] - s[n-2]$$

with $s[-1] = s[-2] = 0$ and $\omega_k = 2\pi k / N$.

**Stage 2 — one complex multiply at the very end (sample N only):**

$$X[k] = s[N-1] - W_N^k \cdot s[N-2]$$

The key insight: the IIR loop uses **only real arithmetic** — one multiply and two additions per sample. The single complex multiply happens just once at the end.

> **Cost**: $N$ real multiplications + 1 complex multiplication, versus $N$ complex multiplications in the naive DFT.

## 4. Reference implementation

In [ ]:
def goertzel(x, k, N=None):
    """Goertzel algorithm — returns complex DFT value at bin k."""
    if N is None:
        N = len(x)
    omega = 2 * np.pi * k / N
    coeff = 2 * np.cos(omega)

    s_prev2 = 0.0  # s[n-2]
    s_prev1 = 0.0  # s[n-1]

    for sample in x:
        s = sample + coeff * s_prev1 - s_prev2
        s_prev2 = s_prev1
        s_prev1 = s

    # Final single complex multiply
    X_k = s_prev1 - np.exp(-1j * omega) * s_prev2
    return X_k


def goertzel_magnitude_sq(x, k, N=None):
    """Return |X[k]|² using only real arithmetic throughout."""
    if N is None:
        N = len(x)
    omega = 2 * np.pi * k / N
    coeff = 2 * np.cos(omega)

    s_prev2 = 0.0
    s_prev1 = 0.0

    for sample in x:
        s = sample + coeff * s_prev1 - s_prev2
        s_prev2 = s_prev1
        s_prev1 = s

    # |X[k]|² = s1² + s2² - coeff·s1·s2  (fully real)
    return s_prev1**2 + s_prev2**2 - coeff * s_prev1 * s_prev2

### Verify against NumPy's FFT

In [ ]:
rng = np.random.default_rng(0)
N = 64
x = rng.standard_normal(N)

fft_result = np.fft.fft(x)

print(f"{'bin':>4}  {'FFT':>30}  {'Goertzel':>30}  {'|error|':>12}")
print("-" * 82)
for k in [0, 1, 7, 16, 31, 63]:
    g = goertzel(x, k, N)
    err = abs(fft_result[k] - g)
    print(f"{k:>4}  {fft_result[k]:>30.6f}  {g:>30.6f}  {err:>12.2e}")

## 5. Hann windowing — the Synple implementation

The bare Goertzel algorithm assumes the input block is **periodic**. When the signal frequency does not land exactly on a bin, the rectangular window (implicit in treating the block as if it repeats) causes **spectral leakage**: energy from a strong tone bleeds into neighbouring bins via high sidelobes (~13 dB below the main lobe).

In Synple's alias detector (`dsp/Goertzel.h`), a **Hann window** is applied to each sample before it enters the IIR:

$$w[i] = \frac{1}{2}\!\left(1 - \cos\!\left(\frac{2\pi i}{N-1}\right)\right)$$

so the recurrence becomes:

$$s[n] = x[n] \cdot w[n] + 2\cos(\omega_k)\,s[n-1] - s[n-2]$$

The Hann window tapers the block smoothly to zero at both ends, pushing the first sidelobe down to ~31 dB. This prevents a large fundamental from registering spurious energy at the alias-detection bin one octave away.

In [ ]:
## 6. Visualising the IIR state evolution

Below we plot $s[n]$, $s[n-1]$, and the running contribution $s[n]^2 + s[n-1]^2 - \text{coeff}\cdot s[n]s[n-1]$ (an approximation of the accumulated power) as we feed a pure tone into the filter.

## 5. Visualising the IIR state evolution

Below we plot $s[n]$, $s[n-1]$, and the running contribution $s[n]^2 + s[n-1]^2 - \text{coeff}\cdot s[n]s[n-1]$ (an approximation of the accumulated power) as we feed a pure tone into the filter.

In [ ]:
## 7. Frequency selectivity — the filter's frequency response

The Goertzel IIR has a **second-order resonator** transfer function:

$$H(z) = \frac{1}{1 - 2\cos(\omega_k)z^{-1} + z^{-2}}$$

It has a pole pair on the unit circle at $\pm\omega_k$. The longer the block size $N$, the narrower its effective bandwidth (because we only care about the accumulated output after exactly $N$ steps).

Below we plot the magnitude response for several target frequencies.

## 6. Frequency selectivity — the filter's frequency response

The Goertzel IIR has a **second-order resonator** transfer function:

$$H(z) = \frac{1}{1 - 2\cos(\omega_k)z^{-1} + z^{-2}}$$

It has a pole pair on the unit circle at $\pm\omega_k$. The longer the block size $N$, the narrower its effective bandwidth (because we only care about the accumulated output after exactly $N$ steps).

Below we plot the magnitude response for several target frequencies.

In [ ]:
## 8. Block size N and frequency resolution

The frequency resolution (bin width) is $\Delta f = f_s / N$. A larger block gives finer resolution but more latency. Here we show how increasing $N$ sharpens the detection of a target tone when a nearby interferer is present.

## 7. Block size N and frequency resolution

The frequency resolution (bin width) is $\Delta f = f_s / N$. A larger block gives finer resolution but more latency. Here we show how increasing $N$ sharpens the detection of a target tone when a nearby interferer is present.

In [ ]:
## 9. DTMF tone detection — a practical example

DTMF (telephone keypad tones) encodes each key as a pair of frequencies:

| | 1209 Hz | 1336 Hz | 1477 Hz | 1633 Hz |
|---|---------|---------|---------|---------|
| **697 Hz** | 1 | 2 | 3 | A |
| **770 Hz** | 4 | 5 | 6 | B |
| **852 Hz** | 7 | 8 | 9 | C |
| **941 Hz** | \* | 0 | \# | D |

We run Goertzel at each of the 8 standard frequencies and pick the highest-power row + column.

## 8. DTMF tone detection — a practical example

DTMF (telephone keypad tones) encodes each key as a pair of frequencies:

| | 1209 Hz | 1336 Hz | 1477 Hz | 1633 Hz |
|---|---------|---------|---------|---------|
| **697 Hz** | 1 | 2 | 3 | A |
| **770 Hz** | 4 | 5 | 6 | B |
| **852 Hz** | 7 | 8 | 9 | C |
| **941 Hz** | \* | 0 | \# | D |

We run Goertzel at each of the 8 standard frequencies and pick the highest-power row + column.

In [ ]:
## 10. Computational cost comparison

When you need $M$ specific bins out of $N$ total, Goertzel beats FFT when $M < N / \log_2 N$.

## 9. Computational cost comparison

When you need $M$ specific bins out of $N$ total, Goertzel beats FFT when $M < N / \log_2 N$.

In [ ]:
## 11. Summary

| Property | Detail |
|---|---|
| **What it computes** | Single DFT bin $X[k]$ |
| **Cost** | $N$ real mults + 1 complex mult |
| **When to use** | Need $M \ll \log_2 N$ specific bins |
| **Core recurrence** | $s[n] = x[n] + 2\cos(\omega_k)\,s[n-1] - s[n-2]$ |
| **Final output** | $X[k] = s[N-1] - e^{-j\omega_k}\,s[N-2]$ |
| **Magnitude only** | $\|X[k]\|^2 = s_1^2 + s_2^2 - 2\cos(\omega_k)\,s_1 s_2$ (fully real) |
| **Windowed variant** | Multiply each input sample by a Hann window before the IIR loop; suppresses sidelobes from ~13 dB to ~31 dB |
| **Classic use-case** | DTMF decoding (8 fixed frequencies) |
| **Caveat** | Non-integer $k$ is fine — just use the exact $\omega_k$ you want |

The algorithm is also directly applicable to **pitch detection** in a synthesizer context: run Goertzel at the expected harmonic frequencies of each MIDI note and pick the note with highest correlated energy — much cheaper than a full FFT when you're tracking a single voice.

## 10. Summary

| Property | Detail |
|---|---|
| **What it computes** | Single DFT bin $X[k]$ |
| **Cost** | $N$ real mults + 1 complex mult |
| **When to use** | Need $M \ll \log_2 N$ specific bins |
| **Core recurrence** | $s[n] = x[n] + 2\cos(\omega_k)\,s[n-1] - s[n-2]$ |
| **Final output** | $X[k] = s[N-1] - e^{-j\omega_k}\,s[N-2]$ |
| **Magnitude only** | $|X[k]|^2 = s_1^2 + s_2^2 - 2\cos(\omega_k)\,s_1 s_2$ (fully real) |
| **Classic use-case** | DTMF decoding (8 fixed frequencies) |
| **Caveat** | Non-integer $k$ is fine — just use the exact $\omega_k$ you want |

The algorithm is also directly applicable to **pitch detection** in a synthesizer context: run Goertzel at the expected harmonic frequencies of each MIDI note and pick the note with highest correlated energy — much cheaper than a full FFT when you're tracking a single voice.